In [ ]:
import os
import pandas as pd
# import utils.age_limit_processing as alp

year = 2015

items_path = f"../data/processed/MAL/items_{year}_enriched.tsv"

risk_items_save_path = f"../data/processed/MAL/risk_items_{year}.tsv"

cover_art_dir = f"../data/processed/MAL/cover_art/"


In [ ]:
# from image_nsfw_dection import FalconsaiNSFWDetector

# detector = FalconsaiNSFWDetector()

# def get_nsfw_score(image_path):
#     return detector.get_nsfw_score(image_path)

In [ ]:
from image_nsfw_dection import LAIONNSFWDetector

nsfw_detector = LAIONNSFWDetector()

def get_nsfw_score(image_path):
    return nsfw_detector.score_image(image_path)

In [ ]:
if os.path.exists(risk_items_save_path):
    items = pd.read_csv(risk_items_save_path, sep="\t")
else:
    items = pd.read_csv(items_path, sep="\t")

In [ ]:

from tqdm.auto import tqdm

for index, row in tqdm(items.iterrows(), total=len(items), desc="Processing items"):
    image_path = os.path.join(cover_art_dir, str(row["item_id"]) + ".jpg")
    if not os.path.exists(image_path):
        print(f"Image for item {row['item_id']} not found at {image_path}. Skipping.")
        items.at[index, "nsfw_score"] = 0
        continue
    nsfw_score = get_nsfw_score(image_path)
    items.at[index, "nsfw_score"] = nsfw_score
    #print(f"Processed item {row['item_id']} with NSFW score: {nsfw_score}")

In [ ]:
items.to_csv(risk_items_save_path, sep="\t", index=False)